In [0]:
USE CATALOG gizmobox;

In [0]:
SELECT * FROM gizmobox.bronze.v_customers;

Databricks data profile. Run in Databricks to view.

## Create customers table 

In [0]:
CREATE TABLE IF NOT EXISTS gizmobox.silver.customers
WITH DIS AS 
(SELECT customer_id,MAX(created_timestamp) AS created_timestamp 
FROM gizmobox.bronze.v_customers WHERE customer_id IS NOT NULL
GROUP BY customer_id)
SELECT DISTINCT 
a.file_path,
CAST(date_format(a.created_timestamp,'yyyy-MM-dd') AS DATE) as created_date,
date_format(a.created_timestamp,'HH:mm:ss') as created_time,
a.customer_id,
a.customer_name,
CAST(a.date_of_birth AS DATE) AS date_of_birth,
a.email,
a.member_since,
a.telephone
FROM gizmobox.bronze.v_customers a INNER JOIN DIS 
ON a.customer_id=DIS.customer_id AND a.created_timestamp=DIS.created_timestamp
WHERE a.customer_id IS NOT NULL;

In [0]:
SELECT * FROM gizmobox.silver.customers;

## Create addresse table

In [0]:
SELECT * FROM gizmobox.bronze.v_addresses;

In [0]:
CREATE TABLE IF NOT EXISTS gizmobox.silver.addresses AS 
SELECT * FROM gizmobox.bronze.v_addresses
pivot (max(address_line_1) as address_line_1,max(city) as city,max(state) as state,max(postcode) as postcode,max(_rescued_data) as _rescued_data for address_type in ('shipping','billing'))


In [0]:
SELECT * FROM gizmobox.silver.addresses;

## Create memebership table

In [0]:
SELECT * FROM gizmobox.bronze.v_memberships;

In [0]:
CREATE TABLE IF NOT EXISTS gizmobox.silver.memberships AS
SELECT cast(regexp_extract(path,'.*/([0-9]+)\\.png$', 1) AS INT) as customer_id,path,modificationTime,length,content from gizmobox.bronze.v_memberships;

In [0]:
SELECT * FROM gizmobox.silver.memberships;

## create orders table

In [0]:
select value:order_id,value from gizmobox.bronze.v_orders

In [0]:
CREATE OR REPLACE TEMP VIEW tv_orders_fixed
AS
SELECT value,
       regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "\$1"') AS fixed_value 
  FROM gizmobox.bronze.v_orders;

In [0]:
SELECT schema_of_json(fixed_value) AS schema,
       fixed_value
  FROM tv_orders_fixed
 LIMIT 1;

In [0]:
SELECT from_json(fixed_value, 
                 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') AS json_value,
       fixed_value
  FROM tv_orders_fixed;

In [0]:
DROP TABLE IF EXISTS gizmobox.silver.orders_json;
CREATE TABLE IF NOT EXISTS gizmobox.silver.orders_json
AS
SELECT from_json(fixed_value, 
                 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') AS json_value
  FROM tv_orders_fixed;

In [0]:
SELECT * FROM gizmobox.silver.orders_json;

In [0]:
 SELECT json_value.order_id,
        json_value.order_status,
        json_value.payment_method,
        json_value.total_amount,
        json_value.transaction_timestamp,
        json_value.customer_id,
        json_value.items
FROM gizmobox.silver.orders_json;

In [0]:
SELECT json_value.order_id,
        json_value.order_status,
        json_value.payment_method,
        json_value.total_amount,
        json_value.transaction_timestamp,
        json_value.customer_id,
        array_distinct(json_value.items) AS items
FROM gizmobox.silver.orders_json;

In [0]:
CREATE OR REPLACE TEMP VIEW tv_orders_exploded
AS
SELECT json_value.order_id,
        json_value.order_status,
        json_value.payment_method,
        json_value.total_amount,
        json_value.transaction_timestamp,
        json_value.customer_id,
        explode(array_distinct(json_value.items)) AS item
FROM gizmobox.silver.orders_json;

In [0]:
CREATE TABLE IF NOT EXISTS gizmobox.silver.orders
AS
SELECT order_id,
       order_status,
       payment_method,
       total_amount,
       transaction_timestamp,
       customer_id,
       item.item_id,
       item.name,
       item.price,
       item.quantity,
       item.category,
       item.details.brand,
       item.details.color
  FROM tv_orders_exploded;

In [0]:
select * from gizmobox.silver.orders;

## CREATE REFUNDS

In [0]:
SELECT * FROM gizmobox.bronze.refunds;

In [0]:
SELECT refund_reason,split(refund_reason,':')[0],split(refund_reason,':')[1] FROM gizmobox.bronze.refunds;

In [0]:
CREATE TABLE gizmobox.silver.refunds
AS
SELECT refund_id,
       payment_id,
       CAST(date_format(refund_timestamp, 'yyyy-MM-dd') AS DATE) AS refund_date,
       date_format(refund_timestamp, 'HH:mm:ss') AS refund_time,
       refund_amount,
       regexp_extract(refund_reason, '^([^:]+):', 1) AS refund_reason,
       regexp_extract(refund_reason, '^[^:]+:(.*)$', 1) AS refund_source
  FROM gizmobox.bronze.refunds;

## CREATE PAYMENTS

In [0]:
CREATE TABLE gizmobox.silver.payments
AS
SELECT payment_id,
       order_id,
       CAST(date_format(payment_date,'yyyy-MM-dd') AS DATE) AS payment_date,
       date_format(payment_date,'HH:mm:ss') AS payment_time,
       CASE payment_status
         WHEN 1 THEN 'Success'
         WHEN 2 THEN 'Pending'
         WHEN 3 THEN 'Cancelled'
         WHEN 4 THEN 'Failed'
       END AS payment_status,  
       payment_method
  FROM gizmobox.bronze.payments;